# 知识型 benchmark 出题：初步结论与完整验证

本入口把研究问题、证据、判据、全部输出和失败边界放在一起。审核由助手完成，**不是人工金标准，也不是正式独立测试集**。

重点：题目隐含的知识后果是否可画；BAGEL 能否利用输入知识；改写／编译是否贡献收益。所有下方图片都是模型输出，不能用作来源事实认证。此轮未提供图像知识、未训练模型、未验证真实检索或统一编码器。

[长期设计约定](/yzp/zhaozy/yangzepeng/0905/demiwtg/curation/DESIGN.md) · [首轮六题及全部失败结果](/yzp/zhaozy/yangzepeng/0905/demiwtg/curation/knowledge_probe.ipynb) · [此前 benchmark200 开发诊断](/yzp/zhaozy/yangzepeng/0905/demiwtg/curation/rag_diagnostic.ipynb)

第一轮为 512/renorm1，参数检查与后续题族另建批次；后续采用 1024/renorm0、50 步、think=False。不能把不同问题与配置的总均值直接比较。

In [ ]:
from pathlib import Path
import sys, json
from IPython.display import HTML, display
root = Path.cwd()
if root.name == 'curation': root = root.parent
assert (root/'curation/DESIGN.md').exists()
if str(root) not in sys.path: sys.path.insert(0,str(root))
from curation.knowledge_probe import render, metrics, summary, collect
color = root/'state/curation/knowledge_color_probe_v1'
transfer = root/'state/curation/knowledge_compiler_probe_v1'
for run in [color, transfer]:
    assert len(summary(run)) == len(collect(run)), 'Review incomplete'
print(json.dumps(json.loads((transfer/'findings.json').read_text()),ensure_ascii=False,indent=2))

## 自动知识编译的场景复验

同一 Qwen 编译提示，分别给正确共享事实、空事实和故意错配事实，输出直接交给 BAGEL，无手工改写。错配事实只是实验负控制，其附带的来源并不支持错误说法。每条件两个新种子；四题组成两个家族。每次文字转换只有一次，图像重复不能反映编译器采样稳定性。

题面仍隐含目标颜色；给 BAGEL 的编译描述显式补出了颜色。因而这是“oracle 资料→文本编译→BAGEL”的系统对照，不能叫 BAGEL 原生 RAG 或微调结果。空资料编译器允许使用自身知识，但本次未补颜色，不代表最强规划基线。

In [ ]:
display(HTML(render(transfer)))
print('编译输入检查：')
print(json.dumps(json.loads((transfer/'compiler_review.json').read_text()),ensure_ascii=False,indent=2))

## 出题与可绘制性检查

4 道颜色题分别比较原题、直接资料拼接、中性场景扩写和助手编写的显式视觉 caption；每条件 3 个配对种子。Gemini 只收到原题，每题生成 3 次。另有腰果与袋熊两道 caption 接口检查，各 9 张 BAGEL 图，没有新增闭源对照。

“知识符合”只表示冻结的视觉属性符合。目标缺席、额外动物或文字、无法观察及材料质感，必须结合逐图记录阅读。

In [ ]:
display(HTML(render(color)))

## 推理配置复核

12 张新生成及 4 张原输出复用，检查 512/1024 与 renorm0/1 的影响。此小节为助手看图复核，未匿名；没有把调参数后的输出覆盖原结果。

In [ ]:
from curation.parameter_probe import render as render_parameters
display(HTML(render_parameters()))